# Messy Mashup — Genre classification (T1-2026)

**Student:** Akash · **Roll No:** 22F2000701

**Kaggle notebook name:** `DL-22F2000701-notebook-t12026`

**Competition:** `jan-2026-dl-gen-ai-project` · **Primary metric:** Macro F1

**Weights & Biases:** project `22f2000701-t12026` (same name as course registration). On Kaggle, add a secret **`wandb_api_key`** (Add-ons → Secrets) with your W&B API key.

| Model | Role |
|-------|------|
| XGBoost + handcrafted features | Classical baseline (third / non-DL path) |
| TinyCNN on mel + deltas | **Built from scratch** |
| EfficientNet-B0 (ImageNet) | **Pretrained** CNN on spectrograms |
| AST | **Pretrained** transformer, fine-tuned |
| Ensemble | Weighted EfficientNet + AST |

**Reproducibility:** `SEED = 42` is set in the imports cell (random, NumPy, PyTorch).



In [ ]:
import os
BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
_checks = [
    ('genres_stems', os.path.isdir(f'{BASE}/genres_stems')),
    ('mashups', os.path.isdir(f'{BASE}/mashups')),
    ('test.csv', os.path.isfile(f'{BASE}/test.csv')),
    ('sample_submission.csv', os.path.isfile(f'{BASE}/sample_submission.csv')),
]
for name, ok in _checks:
    print(f"{name}: {'OK' if ok else 'MISSING'}")



In [2]:
!pip install -q wandb librosa transformers accelerate

In [ ]:
import os, glob, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from tqdm.notebook import tqdm
import wandb

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WANDB_PROJECT = '22f2000701-t12026'

try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret('wandb_api_key')
    wandb.login(key=_wandb_key)
except Exception as e:
    _env = os.environ.get('WANDB_API_KEY')
    if _env:
        wandb.login(key=_env)
    else:
        raise RuntimeError(
            "Add Kaggle secret 'wandb_api_key' or set WANDB_API_KEY in the environment."
        ) from e

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)



In [ ]:
BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUP_DIR  = BASE
ESC_DIR     = f'{BASE}/ESC-50-master/audio'
TEST_CSV    = f'{BASE}/test.csv'
SAMPLE_SUB  = f'{BASE}/sample_submission.csv'
EFF_CACHE   = '/kaggle/working/eff_cache'
AST_CACHE   = '/kaggle/working/ast_cache'
os.makedirs(EFF_CACHE, exist_ok=True)
os.makedirs(AST_CACHE, exist_ok=True)

GENRES   = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
SR       = 22050
SR_AST   = 16000
DURATION = 30
SEG_LEN  = 5
SEG_HOP  = 2
N_MELS   = 128
HOP      = 512
N_MFCC   = 40

le = LabelEncoder()
le.fit(GENRES)

test_df      = pd.read_csv(TEST_CSV)
esc_files    = glob.glob(f'{ESC_DIR}/*.wav')
filename_col = test_df.columns[1]
print(test_df.columns.tolist())
print(test_df.head(3))
print(f"ESC noise files: {len(esc_files)}")
print(f"Sample mashup path: {os.path.join(MASHUP_DIR, test_df.iloc[0][filename_col])}")

In [ ]:
def load_audio(fp, sr=SR, duration=DURATION):
    y, _ = librosa.load(fp, sr=sr, duration=duration, mono=True)
    target = int(sr * duration)
    if len(y) < target:
        y = np.tile(y, int(np.ceil(target / len(y))))[:target]
    return y[:target].astype(np.float32)

def mix_stems(song_path, sr=SR):
    mixed = np.zeros(sr * DURATION, dtype=np.float32)
    for stem in ['drums.wav', 'vocals.wav', 'bass.wav', 'others.wav']:
        fp = os.path.join(song_path, stem)
        if os.path.exists(fp):
            try:
                mixed += load_audio(fp, sr=sr, duration=DURATION)
            except:
                pass
    peak = np.abs(mixed).max()
    if peak > 0:
        mixed /= peak
    return mixed

def add_noise(y, noise_files, sr=SR, snr_db=None):
    if not noise_files:
        return y
    if snr_db is None:
        snr_db = random.uniform(5, 15)
    try:
        noise      = load_audio(random.choice(noise_files), sr=sr, duration=DURATION)
        signal_rms = np.sqrt(np.mean(y**2)) + 1e-9
        noise_rms  = np.sqrt(np.mean(noise**2)) + 1e-9
        noise      = noise * (signal_rms / ((10 ** (snr_db / 20)) * noise_rms))
        start      = random.randint(0, max(0, len(y) - len(noise)))
        chunk_len  = min(len(noise), len(y) - start)
        y          = y.copy()
        y[start:start + chunk_len] += noise[:chunk_len]
    except:
        pass
    return y

def get_segments(y, sr=SR):
    seg_samples = sr * SEG_LEN
    hop_samples = sr * SEG_HOP
    segs = []
    for start in range(0, len(y) - seg_samples + 1, hop_samples):
        segs.append(y[start:start + seg_samples].copy())
    if not segs:
        pad = np.zeros(seg_samples, dtype=np.float32)
        pad[:min(len(y), seg_samples)] = y[:seg_samples]
        segs.append(pad)
    return segs

def to_3ch_mel(y, sr=SR):
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    d1     = librosa.feature.delta(mel_db)
    d2     = librosa.feature.delta(mel_db, order=2)
    out    = np.stack([mel_db, d1, d2], axis=0)
    for i in range(3):
        out[i] = (out[i] - out[i].mean()) / (out[i].std() + 1e-9)
    return out.astype(np.float32)

def spec_augment(mel, freq_mask=20, time_mask=50, n_masks=2):
    mel     = mel.copy()
    _, F, T = mel.shape
    for _ in range(n_masks):
        f  = random.randint(1, max(1, min(freq_mask, F - 1)))
        f0 = random.randint(0, max(0, F - f - 1))
        mel[:, f0:f0 + f, :] = 0
        t  = random.randint(1, max(1, min(time_mask, T - 1)))
        t0 = random.randint(0, max(0, T - t - 1))
        mel[:, :, t0:t0 + t] = 0
    return mel

def extract_handcrafted(y):
    feats    = []
    mfcc     = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC)
    chroma   = librosa.feature.chroma_stft(y=y, sr=SR)
    contrast = librosa.feature.spectral_contrast(y=y, sr=SR)
    zcr      = librosa.feature.zero_crossing_rate(y)
    rms      = librosa.feature.rms(y=y)
    feats.extend([mfcc.mean(1), mfcc.std(1), chroma.mean(1), chroma.std(1),
                  contrast.mean(1), contrast.std(1), zcr.mean(1), rms.mean(1)])
    try:
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=SR)
        feats.extend([tonnetz.mean(1), tonnetz.std(1)])
    except:
        feats.extend([np.zeros(6), np.zeros(6)])
    try:
        tempo = librosa.feature.tempo(y=y, sr=SR)
        feats.append([float(tempo[0])])
    except:
        feats.append([0.0])
    return np.concatenate(feats)

In [ ]:
song_paths, song_labels = [], []
for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)
    for song in sorted(os.listdir(genre_path)):
        sp = os.path.join(genre_path, song)
        if os.path.isdir(sp):
            song_paths.append(sp)
            song_labels.append(genre)

genre_songs = {g: [] for g in GENRES}
for sp, lbl in zip(song_paths, song_labels):
    genre_songs[lbl].append(sp)

print(f"Total songs: {len(song_paths)}")
for g in GENRES:
    print(f"  {g}: {len(genre_songs[g])}")

In [ ]:
print("Pre-caching EfficientNet mel spectrograms...")
eff_index = []

for genre in GENRES:
    label_enc = int(le.transform([genre])[0])
    for song_idx, sp in enumerate(tqdm(genre_songs[genre], desc=genre)):
        try:
            y_full     = mix_stems(sp, sr=SR)
            clean_segs = get_segments(y_full, sr=SR)
            noisy      = add_noise(y_full.copy(), esc_files, sr=SR, snr_db=random.uniform(5, 12))
            noisy_segs = get_segments(noisy, sr=SR)
            for seg_idx, seg in enumerate(clean_segs + noisy_segs):
                mel3       = to_3ch_mel(seg, sr=SR)
                cache_path = f'{EFF_CACHE}/{genre}_{song_idx}_s{seg_idx}.npy'
                np.save(cache_path, mel3)
                eff_index.append({'path': cache_path, 'label': label_enc, 'genre': genre})
        except:
            pass

eff_cache_df = pd.DataFrame(eff_index)
eff_cache_df.to_csv(f'{EFF_CACHE}/index.csv', index=False)
print(f"Total EfficientNet cached segments: {len(eff_cache_df)}")

In [ ]:
class MelDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel = np.load(row['path'])
        if self.augment:
            mel = spec_augment(mel)
            if random.random() < 0.4:
                mel = mel * random.uniform(0.85, 1.15)
        return torch.tensor(mel, dtype=torch.float32), int(row['label'])

train_eff_df, val_eff_df = train_test_split(
    eff_cache_df, test_size=0.15, stratify=eff_cache_df['label'], random_state=42
)
train_eff_ds = MelDataset(train_eff_df, augment=True)
val_eff_ds   = MelDataset(val_eff_df,   augment=False)
train_eff_dl = DataLoader(train_eff_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_eff_dl   = DataLoader(val_eff_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_eff_ds)} | Val: {len(val_eff_ds)}")

In [ ]:
print("XGBoost minimal run...")
run_xgb = wandb.init(project=WANDB_PROJECT, name="xgb_baseline")

X_feat, y_feat = [], []
for genre in GENRES:
    label_enc = int(le.transform([genre])[0])
    for sp in genre_songs[genre][:5]:
        try:
            y = mix_stems(sp)
            X_feat.append(extract_handcrafted(y))
            y_feat.append(label_enc)
        except:
            pass

X_feat = np.array(X_feat)
y_feat = np.array(y_feat)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_feat, y_feat, test_size=0.3, random_state=42)

xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
xgb_model  = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                            eval_metric='mlogloss', device=xgb_device)
xgb_model.fit(X_tr, y_tr)
xgb_preds = xgb_model.predict(X_vl)
f1_xgb    = f1_score(y_vl, xgb_preds, average='macro')
acc_xgb   = accuracy_score(y_vl, xgb_preds)
wandb.log({'val_f1_macro': f1_xgb, 'val_accuracy': acc_xgb})
run_xgb.finish()
print(f"XGBoost F1: {f1_xgb:.4f} | Acc: {acc_xgb:.4f}")

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 16, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

run_cnn   = wandb.init(project=WANDB_PROJECT, name="cnn_scratch")
cnn_model = TinyCNN().to(DEVICE)
opt_cnn   = optim.Adam(cnn_model.parameters(), lr=1e-3)
crit_cnn  = nn.CrossEntropyLoss(label_smoothing=0.1)
f1_cnn    = 0

for epoch in range(1):
    cnn_model.train()
    total_loss = 0
    for xb, yb in tqdm(train_eff_dl, desc=f"CNN Ep {epoch+1}", leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt_cnn.zero_grad()
        loss = crit_cnn(cnn_model(xb), yb)
        loss.backward()
        opt_cnn.step()
        total_loss += loss.item()
    cnn_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in val_eff_dl:
            preds.extend(cnn_model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    f1_cnn  = f1_score(labels, preds, average='macro')
    acc_cnn = accuracy_score(labels, preds)
    wandb.log({'epoch': epoch+1, 'train_loss': total_loss/len(train_eff_dl),
               'val_f1_macro': f1_cnn, 'val_accuracy': acc_cnn})
    print(f"CNN Ep {epoch+1:02d} | Loss: {total_loss/len(train_eff_dl):.4f} | F1: {f1_cnn:.4f}")

wandb.log({'best_val_f1': f1_cnn})
run_cnn.finish()
print(f"CNN final F1: {f1_cnn:.4f}")

In [ ]:
class EfficientNetGenre(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.base = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.base.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1280, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.base(x)

def train_model(model, train_dl, val_dl, epochs, lr, run_name, patience=8):
    run       = wandb.init(project=WANDB_PROJECT, name=run_name,
                           config={'epochs': epochs, 'lr': lr})
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_f1, best_state, no_improve = 0, None, 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in tqdm(train_dl, desc=f"Ep {epoch+1}", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                labels.extend(yb.numpy())

        f1  = f1_score(labels, preds, average='macro')
        acc = accuracy_score(labels, preds)
        wandb.log({'epoch': epoch+1, 'train_loss': total_loss/len(train_dl),
                   'val_f1_macro': f1, 'val_accuracy': acc})
        print(f"Ep {epoch+1:02d} | Loss: {total_loss/len(train_dl):.4f} | F1: {f1:.4f} | Acc: {acc:.4f}")

        if f1 > best_f1:
            best_f1    = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    wandb.log({'best_val_f1': best_f1})
    run.finish()
    return model, best_f1

In [ ]:
eff_model = EfficientNetGenre().to(DEVICE)
eff_model, f1_eff = train_model(eff_model, train_eff_dl, val_eff_dl,
                                 epochs=40, lr=1e-3, run_name='efficientnet_b0_pretrained')
torch.save(eff_model.state_dict(), '/kaggle/working/efficientnet_final.pt')
print(f"EfficientNet best F1: {f1_eff:.4f}")

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

ast_extractor = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')

print("Pre-caching AST features...")
ast_index = []

for genre in GENRES:
    label_enc = int(le.transform([genre])[0])
    for song_idx, sp in enumerate(tqdm(genre_songs[genre], desc=genre)):
        try:
            y = mix_stems(sp, sr=SR_AST)
            for win_idx in range(5):
                seg_samples = SR_AST * 10
                start       = random.randint(0, max(0, len(y) - seg_samples))
                seg         = y[start:start + seg_samples].copy()
                if win_idx >= 3:
                    seg = add_noise(seg, esc_files, sr=SR_AST, snr_db=random.uniform(5, 12))
                inputs     = ast_extractor(seg, sampling_rate=SR_AST, return_tensors='pt',
                                           padding='max_length', max_length=1024)
                feat       = inputs['input_values'].squeeze(0).numpy()
                cache_path = f'{AST_CACHE}/{genre}_{song_idx}_w{win_idx}.npy'
                np.save(cache_path, feat)
                ast_index.append({'path': cache_path, 'label': label_enc, 'genre': genre})
        except:
            pass

ast_cache_df = pd.DataFrame(ast_index)
ast_cache_df.to_csv(f'{AST_CACHE}/index.csv', index=False)
print(f"Total AST cached samples: {len(ast_cache_df)}")

In [ ]:
class ASTDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        feat = np.load(row['path'])
        if self.augment:
            feat = feat.copy()
            t    = random.randint(1, 80)
            t0   = random.randint(0, max(0, feat.shape[0] - t - 1))
            feat[t0:t0 + t, :] = 0
            f    = random.randint(1, 20)
            f0   = random.randint(0, max(0, feat.shape[1] - f - 1))
            feat[:, f0:f0 + f] = 0
        return torch.tensor(feat, dtype=torch.float32), int(row['label'])

train_ast_df, val_ast_df = train_test_split(
    ast_cache_df, test_size=0.15, stratify=ast_cache_df['label'], random_state=42
)
train_ast_ds = ASTDataset(train_ast_df, augment=True)
val_ast_ds   = ASTDataset(val_ast_df,   augment=False)
train_ast_dl = DataLoader(train_ast_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_ast_dl   = DataLoader(val_ast_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f"AST Train: {len(train_ast_ds)} | Val: {len(val_ast_ds)}")

In [ ]:
ast_model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)

for param in ast_model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

run_ast1 = wandb.init(project=WANDB_PROJECT, name='ast_head_only',
                      config={'epochs': 5, 'lr': 1e-3, 'phase': 'head_only'})
opt_ast1 = optim.AdamW(filter(lambda p: p.requires_grad, ast_model.parameters()),
                        lr=1e-3, weight_decay=1e-4)
crit_ast = nn.CrossEntropyLoss(label_smoothing=0.1)

for epoch in range(5):
    ast_model.train()
    total_loss = 0
    for xb, yb in tqdm(train_ast_dl, desc=f'AST P1 Ep {epoch+1}', leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt_ast1.zero_grad()
        loss = crit_ast(ast_model(input_values=xb).logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(ast_model.parameters(), 1.0)
        opt_ast1.step()
        total_loss += loss.item()
    ast_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in val_ast_dl:
            preds.extend(ast_model(input_values=xb.to(DEVICE)).logits.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    f1  = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    wandb.log({'epoch': epoch+1, 'train_loss': total_loss/len(train_ast_dl),
               'val_f1_macro': f1, 'val_accuracy': acc})
    print(f'AST P1 Ep {epoch+1} | Loss: {total_loss/len(train_ast_dl):.4f} | F1: {f1:.4f}')

run_ast1.finish()

In [ ]:
for param in ast_model.audio_spectrogram_transformer.parameters():
    param.requires_grad = True

run_ast2       = wandb.init(project=WANDB_PROJECT, name='ast_full_finetune',
                            config={'epochs': 20, 'lr': 1e-5, 'phase': 'full_finetune'})
opt_ast2       = optim.AdamW(ast_model.parameters(), lr=1e-5, weight_decay=1e-4)
sch_ast        = optim.lr_scheduler.CosineAnnealingLR(opt_ast2, T_max=20)
best_f1_ast    = 0
best_ast_state = None
no_impr_ast    = 0

for epoch in range(20):
    ast_model.train()
    total_loss = 0
    for xb, yb in tqdm(train_ast_dl, desc=f'AST P2 Ep {epoch+1}', leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt_ast2.zero_grad()
        loss = crit_ast(ast_model(input_values=xb).logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(ast_model.parameters(), 1.0)
        opt_ast2.step()
        total_loss += loss.item()
    sch_ast.step()

    ast_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in val_ast_dl:
            preds.extend(ast_model(input_values=xb.to(DEVICE)).logits.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())

    f1  = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    wandb.log({'epoch': epoch+1, 'train_loss': total_loss/len(train_ast_dl),
               'val_f1_macro': f1, 'val_accuracy': acc})
    print(f'AST P2 Ep {epoch+1:02d} | Loss: {total_loss/len(train_ast_dl):.4f} | F1: {f1:.4f}')

    if f1 > best_f1_ast:
        best_f1_ast    = f1
        best_ast_state = {k: v.clone() for k, v in ast_model.state_dict().items()}
        no_impr_ast    = 0
    else:
        no_impr_ast += 1
        if no_impr_ast >= 6:
            print(f'Early stopping at epoch {epoch+1}')
            break

ast_model.load_state_dict(best_ast_state)
wandb.log({'best_val_f1': best_f1_ast})
run_ast2.finish()
torch.save(ast_model.state_dict(), '/kaggle/working/ast_final.pt')
print(f'AST Final best F1: {best_f1_ast:.4f}')

In [ ]:
eff_model.eval()
ast_model.eval()

w_total = f1_eff + best_f1_ast
w_eff   = f1_eff / w_total
w_ast   = best_f1_ast / w_total
print(f"Ensemble weights — EfficientNet: {w_eff:.3f} | AST: {w_ast:.3f}")

def predict_eff(fp):
    probs       = np.zeros(10)
    y           = load_audio(fp, sr=SR, duration=DURATION)
    seg_samples = SR * SEG_LEN
    for _ in range(5):
        start = random.randint(0, max(0, len(y) - seg_samples))
        seg   = y[start:start + seg_samples]
        if random.random() < 0.5:
            seg = np.roll(seg, random.randint(0, SR * 2))
        mel = to_3ch_mel(seg, sr=SR)
        x   = torch.tensor(mel[np.newaxis]).to(DEVICE)
        with torch.no_grad():
            probs += F.softmax(eff_model(x), dim=1).cpu().numpy()[0]
    return probs / 5

def predict_ast_fn(fp):
    probs = np.zeros(10)
    try:
        y           = load_audio(fp, sr=SR_AST, duration=DURATION)
        seg_samples = SR_AST * 10
        for _ in range(3):
            start  = random.randint(0, max(0, len(y) - seg_samples))
            seg    = y[start:start + seg_samples]
            inputs = ast_extractor(seg, sampling_rate=SR_AST, return_tensors='pt',
                                   padding='max_length', max_length=1024)
            x = inputs['input_values'].to(DEVICE)
            with torch.no_grad():
                probs += F.softmax(ast_model(input_values=x).logits, dim=1).cpu().numpy()[0]
        return probs / 3
    except:
        return probs

all_preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    fp = os.path.join(MASHUP_DIR, row[filename_col])
    try:
        p_eff    = predict_eff(fp)
        p_ast    = predict_ast_fn(fp)
        combined = w_eff * p_eff + w_ast * p_ast
        all_preds.append(int(np.argmax(combined)))
    except Exception as e:
        print(f"Error: {fp} | {e}")
        all_preds.append(0)

print(pd.Series(le.inverse_transform(all_preds)).value_counts())

In [ ]:
sub          = pd.read_csv(SAMPLE_SUB)
sub['genre'] = le.inverse_transform(all_preds)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub['genre'].value_counts())
print(sub.head())

compare_df = pd.DataFrame({
    'model': [
        'XGBoost (handcrafted)',
        'TinyCNN (scratch)',
        'EfficientNet-B0 (pretrained)',
        'AST (fine-tuned)',
    ],
    'val_f1_macro': [f1_xgb, f1_cnn, f1_eff, best_f1_ast],
})
print('Validation Macro F1 — model comparison')
print(compare_df.to_string(index=False))

run_final = wandb.init(project=WANDB_PROJECT, name='final_ensemble_submission')
wandb.log({'val_f1_comparison': wandb.Table(dataframe=compare_df)})
wandb.log({
    'xgb_val_f1':          round(f1_xgb, 4),
    'cnn_val_f1':          round(f1_cnn, 4),
    'efficientnet_val_f1': round(f1_eff, 4),
    'ast_val_f1':          round(best_f1_ast, 4),
    'eff_ensemble_weight': round(w_eff, 3),
    'ast_ensemble_weight': round(w_ast, 3)
})
wandb.save('/kaggle/working/submission.csv')
run_final.finish()
print("Submission saved.")

